# Section-D: SQL Advanced Analysis

## E-Commerce Order Analytics System

This notebook performs advanced SQL analysis using Common Table Expressions (CTEs), Window Functions, Ranking Functions, and Time-Series Analysis.

The objective is to derive deeper business insights from customer purchasing behavior, regional performance, and revenue trends.


## Database Connection

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("ecommerce.db")

## Query 1: Running Revenue by Region

This query calculates cumulative revenue for each region over time using window functions.

In [2]:
query = """
SELECT
    o.region_code,
    o.order_date,
    SUM(oi.quantity * oi.unit_price) AS revenue,
    SUM(SUM(oi.quantity * oi.unit_price))
        OVER (
            PARTITION BY o.region_code
            ORDER BY o.order_date
        ) AS running_revenue
FROM orders o
JOIN order_items oi
ON o.order_id = oi.order_id
GROUP BY o.region_code, o.order_date;
"""

pd.read_sql(query, conn)

,region_code,order_date,revenue,running_revenue
0,EAST,2025-08-11 15:14:57,821.92,821.92
1,EAST,2025-08-22 15:14:57,17807.15,18629.07
2,EAST,2025-08-24 15:14:57,32134.32,50763.39
3,EAST,2025-08-27 15:14:57,33055.83,83819.22
4,EAST,2025-08-28 15:14:57,39813.80,123633.02
...,...,...,...,...
416,WEST,2026-07-27 15:14:57,79255.98,2579139.42
417,WEST,2026-07-28 15:14:57,4058.21,2583197.63
418,WEST,2026-07-31 15:14:57,41408.83,2624606.46
419,WEST,2026-08-06 15:14:57,51668.11,2676274.57


## Query 2: Customer Revenue Ranking

Ranks customers based on their total spending.

In [3]:
query = """
WITH customer_revenue AS
(
    SELECT
        c.customer_name,
        SUM(oi.quantity * oi.unit_price) AS revenue
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    JOIN order_items oi
        ON o.order_id = oi.order_id
    GROUP BY c.customer_name
)

SELECT
    customer_name,
    revenue,
    DENSE_RANK()
        OVER(ORDER BY revenue DESC) AS rank_no
FROM customer_revenue;
"""

pd.read_sql(query, conn)

,customer_name,revenue,rank_no
0,Kelly Anderson,161069.43,1
1,Tammy Owens,144113.25,2
2,Joseph Reese,126777.66,3
3,Daniel Nolan,108733.92,4
4,Roger James,99574.88,5
...,...,...,...
287,Taylor Miller,780.29,288
288,Felicia Winters,748.76,289
289,Charles Thomas DDS,-1028.16,290
290,Jason Li,-4844.56,291


## Query 3: Top Customer in Each Region

Identifies the highest spending customer in every region.

In [4]:
query = """
WITH regional_sales AS
(
    SELECT
        o.region_code,
        c.customer_name,
        SUM(oi.quantity * oi.unit_price) AS revenue
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    JOIN order_items oi
        ON o.order_id = oi.order_id
    GROUP BY o.region_code, c.customer_name
)

SELECT *
FROM
(
    SELECT *,
           ROW_NUMBER()
           OVER(
               PARTITION BY region_code
               ORDER BY revenue DESC
           ) AS rn
    FROM regional_sales
)
WHERE rn = 1;
"""

pd.read_sql(query, conn)

,region_code,customer_name,revenue,rn
0,EAST,Carla Long,77810.88,1
1,NORTH,Amy Maynard,72400.68,1
2,SOUTH,Daniel Roberts,75160.93,1
3,WEST,Tammy Owens,96934.87,1


## Query 4: Monthly Revenue Growth

Compares monthly revenue with the previous month using LAG().

In [5]:
query = """
WITH monthly_sales AS
(
    SELECT
        strftime('%Y-%m', o.order_date) AS month,
        SUM(oi.quantity * oi.unit_price) AS revenue
    FROM orders o
    JOIN order_items oi
    ON o.order_id = oi.order_id
    GROUP BY month
)

SELECT
    month,
    revenue,
    LAG(revenue)
    OVER(ORDER BY month) AS previous_month,
    revenue -
    LAG(revenue)
    OVER(ORDER BY month) AS growth
FROM monthly_sales;
"""

pd.read_sql(query, conn)

,month,revenue,previous_month,growth
0,2025-08,634665.26,NaN,NaN
1,2025-09,959321.13,634665.26,324655.87
2,2025-10,1032797.75,959321.13,73476.62
3,2025-11,828468.17,1032797.75,-204329.58
4,2025-12,761126.08,828468.17,-67342.09
5,2026-01,1055077.26,761126.08,293951.18
6,2026-02,1062153.72,1055077.26,7076.46
7,2026-03,695933.60,1062153.72,-366220.12
8,2026-04,1005887.77,695933.60,309954.17
9,2026-05,875246.22,1005887.77,-130641.55


## Query 5: Customer Segmentation using NTILE

Divides customers into four spending groups.

In [6]:
query = """
WITH customer_revenue AS
(
    SELECT
        c.customer_name,
        SUM(oi.quantity * oi.unit_price) AS revenue
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    JOIN order_items oi
        ON o.order_id = oi.order_id
    GROUP BY c.customer_name
)

SELECT
    customer_name,
    revenue,
    NTILE(4)
    OVER(ORDER BY revenue DESC) AS customer_segment
FROM customer_revenue;
"""

pd.read_sql(query, conn)

,customer_name,revenue,customer_segment
0,Kelly Anderson,161069.43,1
1,Tammy Owens,144113.25,1
2,Joseph Reese,126777.66,1
3,Daniel Nolan,108733.92,1
4,Roger James,99574.88,1
...,...,...,...
287,Taylor Miller,780.29,4
288,Felicia Winters,748.76,4
289,Charles Thomas DDS,-1028.16,4
290,Jason Li,-4844.56,4


## Query 6: First and Last Purchase Date

Identifies the first and most recent order date for every customer.

In [7]:
query = """
SELECT
    c.customer_name,
    MIN(o.order_date) AS first_purchase,
    MAX(o.order_date) AS last_purchase
FROM customers c
JOIN orders o
ON c.customer_id = o.customer_id
GROUP BY c.customer_name;
"""

pd.read_sql(query, conn)

,customer_name,first_purchase,last_purchase
0,Aaron Black,2026-02-26 15:14:57,2026-05-28 15:14:57
1,Adam Marshall,2026-02-24 15:14:57,2026-02-24 15:14:57
2,Albert Stewart,2026-06-05 15:14:57,2026-06-10 15:14:57
3,Alex Le,2025-08-09 15:14:57,2025-08-09 15:14:57
4,Allison Flowers,2026-02-24 15:14:57,2026-02-24 15:14:57
...,...,...,...
294,Wanda Cross,2025-10-21 15:14:57,2025-10-21 15:14:57
295,William Gomez,2025-10-16 15:14:57,2026-06-18 15:14:57
296,William Sanchez,2025-11-21 15:14:57,2025-11-21 15:14:57
297,Willie Patterson,2026-02-25 15:14:57,2026-05-24 15:14:57


# Key Learnings

During this advanced SQL analysis, the following concepts were implemented:

- Common Table Expressions (CTEs)
- Window Functions
- DENSE_RANK()
- ROW_NUMBER()
- LAG()
- NTILE()
- Running Totals
- Customer Segmentation
- Revenue Trend Analysis
- Regional Performance Analysis

These techniques are widely used in modern analytics and business intelligence systems.

---

# Conclusion

This notebook demonstrated advanced SQL analytics techniques on the e-commerce dataset. By leveraging CTEs and Window Functions, deeper insights were extracted regarding customer behavior, revenue growth, purchasing patterns, and regional performance.

These analyses provide valuable information for business decision-making and represent real-world SQL practices commonly used in data engineering and business intelligence projects.